# Clase 003 — Git y GitHub para data scientists

**Parte 0 — Prerrequisitos** · Pro Git caps. 2-3.

> 🎯 Usar git como sistema serio de versionado: commits atómicos, branches, PRs, conflictos sin pánico, `.gitignore` para DS.

> ⏱️ ~120 min

## ⚙️ Setup

La mayoría de los ejercicios se hacen en terminal. Este notebook documenta los comandos y verifica el estado del repo desde Python.

In [ ]:
import subprocess
from pathlib import Path

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.stdout.strip() or r.stderr.strip()

print('git version:', run('git --version'))
print('cwd        :', Path.cwd())

## 1️⃣ Modelo mental de git

```
[working tree]  ──git add──▶  [staging area]  ──git commit──▶  [local repo]  ──git push──▶  [remote]
     editas                    preparas                       guardas                    publicas
```

Cada commit es **un nodo en un DAG** identificado por su SHA-1. Las ramas son **punteros móviles** a commits. `HEAD` es el puntero al commit actual.

In [ ]:
# Inspecciona el estado del repo donde corre este notebook
print('--- branch actual ---')
print(run('git branch --show-current'))
print('--- últimos 5 commits ---')
print(run('git log --oneline -5'))
print('--- archivos modificados ---')
print(run('git status -s') or '(working tree limpio)')

## 2️⃣ Commits atómicos + mensajes convencionales

**Atómico** = un commit = un cambio lógico que puede revertirse solo.

**Mensaje convencional**: `tipo(scope): descripción corta`

Tipos comunes:
- `feat` — nueva funcionalidad
- `fix` — bugfix
- `docs` — solo documentación
- `refactor` — refactor sin cambio de comportamiento
- `test` — añade/modifica tests
- `chore` — mantenimiento (deps, build)

Ejemplos buenos:
- `feat(api): agregar endpoint /predict para modelo v2`
- `fix(loader): manejar nulos en columna fecha`
- `docs(readme): aclarar requisitos de instalación`

Ejemplos malos:
- `update` ← ¿qué?
- `fix bug` ← ¿cuál bug?
- `wip` ← no llega a main

## 3️⃣ Branches: crear, mergear, conflicto

```bash
git switch -c feature/nuevo-modelo    # crea y cambia a nueva rama
# … editas, commits …
git switch main
git merge feature/nuevo-modelo        # merge
```

**Conflicto** = git no puede decidir qué versión gana en una línea. Git marca el archivo así:

```
<<<<<<< HEAD
versión de main
=======
versión de la rama
>>>>>>> feature/nuevo-modelo
```

**Resolución**: edita el archivo dejando solo lo que quieres, borra los marcadores, `git add <archivo>`, `git commit` (mensaje pre-rellenado).

**Merge vs rebase** (regla simple): merge para ramas compartidas, rebase solo para tu rama local antes de PR.

## 4️⃣ `.gitignore` para data science

Las reglas que **siempre** van en un proyecto de DS:

```gitignore
# Entornos
.venv/
venv/
__pycache__/
*.pyc

# Notebooks
.ipynb_checkpoints/
# (opcional) limpiar outputs: usa nbstripout en pre-commit

# Datos
data/raw/*
data/interim/*
!data/raw/.gitkeep
!data/interim/.gitkeep

# Modelos y artefactos
models/*.pkl
models/*.joblib
*.h5

# Secretos
.env
.env.*
!.env.example

# IDE
.vscode/
.idea/
.DS_Store
```

**Regla de oro:** todo lo que pese >100 MB o sea sensible NUNCA al repo. Para datos versionados usa DVC (clase 159).

In [ ]:
# Demo: simular qué se commitearía
ignored_examples = ['.venv/lib/site-packages/numpy.py', 'data/raw/customers.csv', '.env', 'models/v2.pkl', '.DS_Store']
respected = ['src/loader.py', 'README.md', 'tests/test_loader.py', 'data/raw/.gitkeep']

print('❌ Estos NO deben aparecer en git status:')
for f in ignored_examples:
    print(f'   {f}')
print()
print('✅ Estos SÍ deben aparecer:')
for f in respected:
    print(f'   {f}')

## 5️⃣ Pull Requests con `gh`

```bash
# Una vez por máquina
gh auth login

# En tu repo, después de push
gh pr create --title "feat: nuevo modelo de churn" --body "## Resumen\n- ..."
gh pr list
gh pr view 12 --web
gh pr merge 12 --squash
```

El PR es donde ocurre la **revisión técnica**. Un PR bueno:
- Hace UNA cosa.
- Tiene descripción del *por qué* (el qué ya está en el diff).
- Pasa CI antes de pedir review.
- Incluye screenshots / outputs si es visual.

## 6️⃣ `git reflog` — la red de seguridad

**Borraste una rama por error. Tranquilo.** Git guarda referencias al HEAD durante ~90 días:

```bash
git reflog                           # lista todo lo que ha sido HEAD
# Encuentra el SHA de tu commit perdido
git switch -c rescate <sha>           # nueva rama desde ese punto
```

Mientras no hayas hecho `git gc` agresivo, casi nada se pierde.

In [ ]:
# Demo: muestra tus últimas 5 entradas del reflog
print(run('git reflog -5'))

## ✅ Checklist

- [ ] Mis commits son atómicos y tienen mensajes convencionales
- [ ] Sé crear branches, mergear y resolver un conflicto
- [ ] Mi `.gitignore` cubre `.venv/`, datos, secrets y outputs
- [ ] Sé abrir y mergear un PR con `gh`
- [ ] Sé que `git reflog` existe y para qué sirve

## 📝 Homework

Ver `README.md`. Repo público en GitHub con 5+ commits convencionales, branch mergeada, `.gitignore` de DS y 1 PR cerrado.

## 📖 Definiciones y características

**Repositorio (repo)**

Carpeta con un subdirectorio `.git/` que guarda toda la historia. Características: contenido inmutable identificado por SHA-1, ramas son punteros móviles, todo cambio publicado es eterno (aunque borres el commit, vive en reflog 90 días).

**Commit**

Snapshot inmutable del estado del repo en un momento. Tiene SHA-1, padre(s), autor, fecha, mensaje. Característica: **atómico** — debería poder revertirse solo sin romper nada.

**Branch (rama)**

Puntero móvil a un commit. Mover el puntero es barato. `HEAD` apunta a la rama actual. La rama `main` no es especial; solo es la rama por defecto del proyecto.

**Working tree / Staging / Repo / Remote**

Las 4 zonas: working tree (lo que editas) → staging area (lo preparado con `git add`) → repo local (lo commiteado) → remote (GitHub/GitLab). Cada `git` mueve cosas entre estas 4 zonas.

**Merge vs Rebase**

**Merge** crea un commit nuevo que junta dos historias (preserva ambas). **Rebase** reescribe los commits de tu rama encima de otra (historia lineal pero modificada). Característica clave: **nunca rebases ramas compartidas** — reescribir SHAs rompe a tus compañeros.

**Conventional Commits**

Convención que prescribe `tipo(scope): descripción`. Tipos: `feat`, `fix`, `docs`, `refactor`, `test`, `chore`, `perf`, `style`. Beneficio: changelogs y semver automáticos.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| `error: failed to push some refs to 'origin/main'` | El remote tiene commits que no tienes localmente (alguien más empujó). **Fix**: `git pull --rebase` primero, resuelve conflictos si los hay, luego `git push`. |
| `fatal: refusing to merge unrelated histories` | Estás juntando dos repos sin ancestro común. **Fix**: `git pull --allow-unrelated-histories` (raro, asegúrate de que es lo que querías). |
| Hice `git reset --hard` y perdí mi trabajo 😱 | Si fue local y no había commit: perdido. Si había commit, **`git reflog`** lo recupera: busca el SHA antes del reset y `git reset --hard <sha>`. |
| `Please tell me who you are` al hacer commit | Falta config global. **Fix**: `git config --global user.name "Tu Nombre"` y `git config --global user.email "tu@email.com"`. |
| Commit con archivo enorme; ahora `git push` rechaza por >100 MB | GitHub bloquea blobs >100 MB. **Fix**: NO basta con borrar el archivo en un commit nuevo (queda en historia). Usa `git filter-repo` o BFG para reescribir historia, o agrega a `.gitignore` desde el inicio. |
| Mergeé un PR pero ahora hay conflictos en `main` | Alguien mergeó algo antes y tu base local es vieja. **Fix**: `git switch main && git pull` y resuelve los conflictos en una nueva rama, no directo en main. |
| `.gitignore` no funciona — el archivo sigue apareciendo en `git status` | Si el archivo **ya estaba trackeado** antes del `.gitignore`, git lo sigue viendo. **Fix**: `git rm --cached <archivo>` y commitea — desde ahora lo ignora. |

## ❓ Preguntas frecuentes

**❓ ¿Merge o rebase?**

Regla simple: **merge para todo lo público, rebase solo localmente antes de PR** para limpiar tus propios commits. Nunca rebases una rama que alguien más usa.

**❓ ¿Force push (`git push -f`) es siempre malo?**

En `main` o ramas compartidas: catástrofe. En tu propia rama de feature después de rebase: aceptable. Mejor usar `--force-with-lease` que falla si alguien más empujó mientras.

**❓ ¿Cómo deshago el último commit?**

Si NO empujaste: `git reset --soft HEAD~1` (mantiene cambios staged) o `--hard` (los borra). Si YA empujaste y quieres revertirlo sin reescribir historia: `git revert HEAD` (crea commit nuevo que deshace).

**❓ ¿Squash o no squash al mergear?**

Squash = un solo commit final con todo el PR. Bueno para mantener historia limpia en main. Pierdes el detalle de pasos intermedios. Política común: squash en PRs pequeños, merge commit en grandes.

**❓ ¿Está bien commitear el `.venv/` o el `data/raw/customers.csv`?**

NO. `.venv/` se reconstruye con `requirements.txt`. Datos grandes/sensibles van fuera del repo (DVC, S3, etc.) — ver clase 159 Parte 4.

**❓ Tengo 30 commits "wip" en mi rama, ¿qué hago antes del PR?**

`git rebase -i main` para entrar al rebase interactivo. Cambia `pick` por `squash` (o `fixup`) en los commits intermedios; quedará un historial limpio.

## 🔗 Referencias

- [Pro Git book](https://git-scm.com/book) — gratis online
- [Conventional Commits](https://www.conventionalcommits.org/)

➡️ **Siguiente:** [004 — Estructura reproducible de proyecto](../004-estructura-reproducible-de-proyecto-cookiecutter-data-science/README.md)

## ✅ Soluciones de los ejercicios

Intentá resolverlos vos primero en tu terminal; acá tenés una solución de referencia comentada.
Cada una corre **git de verdad dentro de un repo temporal** (`tempfile.TemporaryDirectory`) que se borra solo,
así el notebook es reproducible sin tocar tu repo ni necesitar red/GitHub. Los ejercicios que requieren
GitHub (PR con `gh`) se **adaptan** con un remoto local `--bare`.

**Utilidad compartida.** Un helper `git()` para correr comandos git contra un repo, con identidad y config
locales (sin depender de tu `~/.gitconfig`).

In [ ]:
import subprocess, sys
from pathlib import Path

def git(repo, *args, check=True):
    """Corre 'git <args>' dentro de <repo> con identidad/config aisladas y devuelve stdout."""
    base = [
        'git',
        '-c', 'user.name=DS Lab', '-c', 'user.email=ds@lab.local',
        '-c', 'init.defaultBranch=main', '-c', 'commit.gpgsign=false',
        '-c', 'core.autocrlf=false', '-c', 'advice.detachedHead=false',
        '-C', str(repo),
    ]
    r = subprocess.run(base + list(args), capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError('git ' + ' '.join(args) + '\n' + r.stderr)
    return r.stdout

print('git version:', subprocess.run(['git', '--version'], capture_output=True, text=True).stdout.strip())
print('helper git() listo.')

**Ejercicio 1.** Repo desde cero: `git init`, 3 archivos, 3 commits con mensajes convencionales (`feat`/`docs`/`chore`).

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    repo = Path(tmp)
    git(repo, 'init', '-q')

    # commit 1: README (docs)
    (repo / 'README.md').write_text('# Mi proyecto DS\n', encoding='utf-8')
    git(repo, 'add', 'README.md'); git(repo, 'commit', '-q', '-m', 'docs: agrega README inicial')

    # commit 2: dataset (feat / chore)
    (repo / 'data.csv').write_text('a,b\n1,2\n', encoding='utf-8')
    git(repo, 'add', 'data.csv'); git(repo, 'commit', '-q', '-m', 'chore: agrega dataset de ejemplo')

    # commit 3: notebook (feat)
    (repo / 'notebook.ipynb').write_text('{}', encoding='utf-8')
    git(repo, 'add', 'notebook.ipynb'); git(repo, 'commit', '-q', '-m', 'feat: agrega notebook de exploracion')

    log = git(repo, 'log', '--oneline').strip().splitlines()
    print('Historial (git log --oneline):')
    for linea in log:
        print('   ', linea)
    assert len(log) == 3, 'Deberian existir exactamente 3 commits'

print('\nOK: 3 commits atomicos con mensajes en formato  tipo: descripcion.')

**Ejercicio 2.** Branch + conflicto: modificá la misma línea en dos ramas, mergeá y resolvé el conflicto a mano.

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    repo = Path(tmp)
    git(repo, 'init', '-q')
    archivo = repo / 'README.md'
    archivo.write_text('titulo: version original\n', encoding='utf-8')
    git(repo, 'add', 'README.md'); git(repo, 'commit', '-q', '-m', 'docs: version original')

    # rama feature/x cambia la linea
    git(repo, 'switch', '-q', '-c', 'feature/x')
    archivo.write_text('titulo: version de la rama feature\n', encoding='utf-8')
    git(repo, 'commit', '-q', '-am', 'docs: cambio en feature')

    # main cambia la MISMA linea distinto
    git(repo, 'switch', '-q', 'main')
    archivo.write_text('titulo: version de main\n', encoding='utf-8')
    git(repo, 'commit', '-q', '-am', 'docs: cambio en main')

    # merge -> conflicto
    res = git(repo, 'merge', 'feature/x', check=False)
    contenido = archivo.read_text(encoding='utf-8')
    print('Hay marcadores de conflicto?', '<<<<<<<' in contenido)
    assert '<<<<<<<' in contenido, 'Deberia haber conflicto en la misma linea'

    # resolvemos a mano: elegimos una version combinada y limpiamos los marcadores
    archivo.write_text('titulo: version consolidada (main + feature)\n', encoding='utf-8')
    git(repo, 'add', 'README.md')
    git(repo, 'commit', '-q', '--no-edit')

    log = git(repo, 'log', '--oneline').strip().splitlines()
    print('\nTras resolver, ultimos commits:')
    for linea in log[:4]:
        print('   ', linea)
    assert '<<<<<<<' not in archivo.read_text(encoding='utf-8')

print('\nOK: el conflicto se resuelve editando el archivo, quitando <<< === >>>, add y commit.')

**Ejercicio 3.** `.gitignore` profesional para DS: verificá con `git status` que los archivos ignorados no aparecen.

In [ ]:
import tempfile
from pathlib import Path

gitignore = '''.venv/
__pycache__/
.ipynb_checkpoints/
data/raw/*.csv
.env
models/*.pkl
'''

with tempfile.TemporaryDirectory() as tmp:
    repo = Path(tmp)
    git(repo, 'init', '-q')
    (repo / '.gitignore').write_text(gitignore, encoding='utf-8')

    # Creamos archivos que DEBERIAN ser ignorados
    (repo / '.venv').mkdir(); (repo / '.venv' / 'x.py').write_text('x', encoding='utf-8')
    (repo / '.env').write_text('SECRET=123', encoding='utf-8')
    (repo / 'data' / 'raw').mkdir(parents=True)
    (repo / 'data' / 'raw' / 'big.csv').write_text('a,b\n1,2', encoding='utf-8')
    (repo / 'models').mkdir(); (repo / 'models' / 'm.pkl').write_text('bin', encoding='utf-8')

    # git status --porcelain: solo deberia listar .gitignore, NO los ignorados
    status = git(repo, 'status', '--porcelain')
    listados = [l[3:] for l in status.splitlines()]
    print('Archivos que git VE (no ignorados):', listados)

    for ignorado in ['.env', 'data/raw/big.csv', 'models/m.pkl']:
        assert not any(ignorado in l for l in listados), f'{ignorado} deberia estar ignorado'
    print('\nOK: .venv/, .env, data/raw/*.csv y models/*.pkl quedan fuera del control de versiones.')

**Ejercicio 4.** PR desde la CLI con `gh`. (Adaptación sin red: simulamos el flujo con un remoto local
`--bare`; el PR abarca exactamente los commits `main..feature`, que es lo que `gh pr create` publica.)

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    # 'GitHub' local: un repo bare hace de remoto
    remoto = tmp / 'remote.git'
    git(tmp, 'init', '-q', '--bare', str(remoto))

    # repo de trabajo
    repo = tmp / 'trabajo'
    repo.mkdir()
    git(repo, 'init', '-q')
    (repo / 'README.md').write_text('# proyecto\n', encoding='utf-8')
    git(repo, 'add', '.'); git(repo, 'commit', '-q', '-m', 'docs: init')
    git(repo, 'remote', 'add', 'origin', str(remoto))
    git(repo, 'push', '-q', 'origin', 'main')

    # rama de feature con un commit -> lo que iria en el PR
    git(repo, 'switch', '-q', '-c', 'feature/nueva-metrica')
    (repo / 'metric.py').write_text('def f1(): ...\n', encoding='utf-8')
    git(repo, 'add', '.'); git(repo, 'commit', '-q', '-m', 'feat: agrega metrica f1')
    git(repo, 'push', '-q', 'origin', 'feature/nueva-metrica')

    # El PR abarca:  main..feature
    rango = git(repo, 'log', '--oneline', 'main..feature/nueva-metrica').strip()
    print('Commits que incluiria el PR (main..feature):')
    print('   ', rango)
    ramas_remotas = git(repo, 'branch', '-r').strip()
    print('\nRamas en el remoto:', ramas_remotas.replace(chr(10), ', '))
    assert 'feat: agrega metrica f1' in rango
    assert 'feature/nueva-metrica' in ramas_remotas

print('\nOK: en GitHub real harias  gh pr create --fill  desde la rama pusheada.')

**Ejercicio 5.** Recuperación con `git reflog`: borrá una rama con commits y recuperala por su SHA.

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    repo = Path(tmp)
    git(repo, 'init', '-q')
    (repo / 'a.txt').write_text('1', encoding='utf-8')
    git(repo, 'add', '.'); git(repo, 'commit', '-q', '-m', 'docs: base')

    # rama con trabajo valioso
    git(repo, 'switch', '-q', '-c', 'experimento')
    (repo / 'b.txt').write_text('resultado importante', encoding='utf-8')
    git(repo, 'add', '.'); git(repo, 'commit', '-q', '-m', 'feat: experimento clave')
    sha = git(repo, 'rev-parse', 'HEAD').strip()
    print('SHA del commit valioso:', sha[:10])

    # BORRAMOS la rama (por error!)
    git(repo, 'switch', '-q', 'main')
    git(repo, 'branch', '-q', '-D', 'experimento')
    print('Rama experimento borrada.')

    # RESCATE: reflog conserva el SHA aunque la rama ya no exista
    reflog = git(repo, 'reflog')
    assert sha[:7] in reflog or 'experimento' in reflog
    git(repo, 'branch', 'rescate', sha)     # recreamos una rama en ese commit
    recuperado = git(repo, 'log', '--oneline', 'rescate').strip()
    print('\nRama de rescate creada; su historial:')
    for l in recuperado.splitlines():
        print('   ', l)
    assert 'feat: experimento clave' in recuperado

print('\nOK: git reflog es la red de seguridad: los commits viven ~90 dias aunque borres la rama.')